[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/02_Introduction_to_ONNX/04_Installation_and_Setup/Installation_and_Setup_Apply.ipynb)

# 2.4 Installation and Setup — Hands-On Practice

## Objective

Verify your ONNX environment, explore version details, run the
**verification script**, and build a minimal end-to-end pipeline.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | Setup | Install and import |
| 2 | Exercise 1: Version Check | All package versions |
| 3 | Exercise 2: Verification Script | Run verify_installation.py in-notebook |
| 4 | Exercise 3: Operator Exploration | Available operators at current opset |
| 5 | Exercise 4: EP Capabilities | Test execution providers |
| 6 | Exercise 5: Build + Run Pipeline | Minimal model from scratch |
| 7 | Exercise 6: Troubleshooting Guide | Common issues and fixes |
| 8 | Challenge: Environment Report | Exportable setup summary |

## Section 1: Setup

In [ ]:
# Uncomment to install:
# !pip install onnx onnxruntime numpy

import sys
import platform
import numpy as np
import os
import time

print(f'Python: {sys.version}')
print(f'NumPy:  {np.__version__}')
print(f'OS:     {platform.system()} {platform.release()}')
print(f'Arch:   {platform.machine()}')

## Section 2: Exercise 1 — Comprehensive Version Check

In [ ]:
packages = [
    ('onnx', 'onnx'),
    ('onnxruntime', 'onnxruntime'),
    ('numpy', 'numpy'),
    ('protobuf', 'google.protobuf'),
    ('onnxoptimizer', 'onnxoptimizer'),
    ('torch', 'torch'),
    ('tensorflow', 'tensorflow'),
    ('sklearn', 'sklearn'),
    ('skl2onnx', 'skl2onnx'),
    ('tf2onnx', 'tf2onnx'),
    ('onnxmltools', 'onnxmltools'),
]

print(f'{"Package":25s} {"Status":12s} {"Version"}')
print('-' * 55)
for name, module_path in packages:
    try:
        mod = __import__(module_path)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'{name:25s} {"installed":12s} {ver}')
    except ImportError:
        print(f'{name:25s} {"not found":12s} -')

# ONNX-specific details
import onnx
from onnx.defs import onnx_opset_version
import onnxruntime as ort

print(f'\nONNX Details:')
print(f'  Default opset version: {onnx_opset_version()}')
print(f'  IR version:            {onnx.IR_VERSION}')
print(f'  ORT available EPs:     {ort.get_available_providers()}')

## Section 3: Exercise 2 — Verification Script (In-Notebook)

The `verify_installation.py` script builds a minimal $Y = X + B$ model
and runs it. Here we reproduce it step-by-step.

In [ ]:
from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model
from onnx.numpy_helper import from_array

print('=== Running Verification Pipeline ===')
print()

# Step 1: Build Y = X + B
B_data = np.array([10.0, 20.0, 30.0, 40.0], dtype=np.float32)
B_init = from_array(B_data, name='B')

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])

graph = make_graph(
    [make_node('Add', ['X', 'B'], ['Y'])],
    'verify', [X], [Y], initializer=[B_init])
model = make_model(graph, opset_imports=[make_opsetid('', 17)])
print('Step 1: Model built')

# Step 2: Validate
check_model(model)
print('Step 2: Model validated (check_model passed)')

# Step 3: Serialize
model_bytes = model.SerializeToString()
print(f'Step 3: Serialized ({len(model_bytes)} bytes)')

# Step 4: Run with ORT
sess = ort.InferenceSession(model_bytes, providers=['CPUExecutionProvider'])
x_input = np.array([[1, 2, 3, 4]], dtype=np.float32)
result = sess.run(None, {'X': x_input})[0]
expected = x_input + B_data

print(f'Step 4: Inference complete')
print(f'  Input:    {x_input.tolist()}')
print(f'  Bias:     {B_data.tolist()}')
print(f'  Output:   {result.tolist()}')
print(f'  Expected: {expected.tolist()}')

# Step 5: Verify
assert np.allclose(result, expected), 'Verification FAILED!'
print('Step 5: Numerical verification PASSED')
print()
print('All checks passed — ONNX installation is working correctly!')

## Section 4: Exercise 3 — Operator Exploration

Explore which operators are available at the current opset version.

In [ ]:
from onnx.defs import get_all_schemas_with_history, get_schema

# Get all current operators
current_opset = onnx_opset_version()
all_schemas = get_all_schemas_with_history()

# Filter to latest version of each op in default domain
current_ops = {}
for s in all_schemas:
    if s.domain == '' and s.since_version <= current_opset:
        if s.name not in current_ops or s.since_version > current_ops[s.name]:
            current_ops[s.name] = s.since_version

print(f'Operators available at opset {current_opset}: {len(current_ops)}')
print('\nAlphabetical listing:')
print('-' * 60)

ops_sorted = sorted(current_ops.items())
# Print in columns
cols = 3
for i in range(0, len(ops_sorted), cols):
    row = ops_sorted[i:i+cols]
    parts = [f'{name:20s}(v{sv:2d})' for name, sv in row]
    print('  ' + '  '.join(parts))

# Most common ops by usage in typical models
common_ops = [
    'MatMul', 'Add', 'Relu', 'Conv', 'BatchNormalization',
    'Softmax', 'Reshape', 'Transpose', 'Gather', 'Concat',
]
print(f'\nKey operators for deep learning:')
for op in common_ops:
    if op in current_ops:
        schema = get_schema(op)
        print(f'  {op:25s} since v{current_ops[op]:2d}  '
              f'inputs={len(schema.inputs)} outputs={len(schema.outputs)}')

## Section 5: Exercise 4 — Execution Provider Capabilities

In [ ]:
# Build a slightly larger model to test EP capabilities
np.random.seed(42)

W = from_array(np.random.randn(4, 4).astype(np.float32), 'W')
b = from_array(np.zeros(4, dtype=np.float32), 'b')

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 4])

g = make_graph(
    [
        make_node('MatMul', ['X', 'W'], ['XW']),
        make_node('Add', ['XW', 'b'], ['Z']),
        make_node('Relu', ['Z'], ['Y']),
    ],
    'ep_test', [X], [Y], [W, b])
model_ep = make_model(g, opset_imports=[make_opsetid('', 17)])
model_bytes = model_ep.SerializeToString()

x_test = np.random.randn(10, 4).astype(np.float32)

print(f'{"EP":35s} {"Status":8s} {"Latency":>10s}')
print('-' * 55)

for ep in ort.get_available_providers():
    try:
        sess = ort.InferenceSession(model_bytes, providers=[ep])
        # Warmup
        sess.run(None, {'X': x_test})
        # Benchmark
        times = []
        for _ in range(200):
            t0 = time.perf_counter()
            sess.run(None, {'X': x_test})
            times.append((time.perf_counter() - t0) * 1e6)
        avg = np.mean(times)
        print(f'{ep:35s} {"OK":8s} {avg:>8.1f} us')
    except Exception as e:
        print(f'{ep:35s} {"FAIL":8s} {str(e)[:30]}')

## Section 6: Exercise 5 — Full Build + Run Pipeline

Build a 2-layer MLP from scratch, save to disk, load, and verify.

In [ ]:
np.random.seed(0)

# 2-layer MLP: 4 → 8 → 2
W1 = from_array(
    (np.random.randn(4, 8) * np.sqrt(2.0 / 4)).astype(np.float32), 'W1')
b1 = from_array(np.zeros(8, dtype=np.float32), 'b1')
W2 = from_array(
    (np.random.randn(8, 2) * np.sqrt(2.0 / 8)).astype(np.float32), 'W2')
b2 = from_array(np.zeros(2, dtype=np.float32), 'b2')

X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 2])

nodes = [
    make_node('MatMul', ['X', 'W1'], ['XW1']),
    make_node('Add', ['XW1', 'b1'], ['Z1']),
    make_node('Relu', ['Z1'], ['H']),
    make_node('MatMul', ['H', 'W2'], ['HW2']),
    make_node('Add', ['HW2', 'b2'], ['Y']),
]

graph = make_graph(nodes, 'mlp', [X], [Y], initializer=[W1, b1, W2, b2])
model_mlp = make_model(graph, opset_imports=[make_opsetid('', 17)])
model_mlp.producer_name = 'ONNX Tutorial'
model_mlp.doc_string = 'Verification MLP: 4→8→2 with ReLU'

# Validate
check_model(model_mlp)

# Shape inference
from onnx import shape_inference
model_mlp = shape_inference.infer_shapes(model_mlp)

# Save
onnx.save(model_mlp, 'verify_mlp.onnx')
fsize = os.path.getsize('verify_mlp.onnx')
print(f'Saved verify_mlp.onnx ({fsize:,} bytes)')

# Load and run
loaded = onnx.load('verify_mlp.onnx')
check_model(loaded)

sess = ort.InferenceSession('verify_mlp.onnx', providers=['CPUExecutionProvider'])

# Test with multiple batch sizes
from onnx.numpy_helper import to_array
W1_np = to_array(loaded.graph.initializer[0])
b1_np = to_array(loaded.graph.initializer[1])
W2_np = to_array(loaded.graph.initializer[2])
b2_np = to_array(loaded.graph.initializer[3])

for bs in [1, 5, 20, 100]:
    x = np.random.randn(bs, 4).astype(np.float32)
    y_onnx = sess.run(None, {'X': x})[0]
    y_np = np.maximum(0, x @ W1_np + b1_np) @ W2_np + b2_np
    match = np.allclose(y_onnx, y_np, atol=1e-6)
    print(f'  batch={bs:4d}  output={y_onnx.shape}  match={match}')

os.remove('verify_mlp.onnx')
print('\nFull pipeline verified!')

## Section 7: Exercise 6 — Troubleshooting Guide

Common installation issues and how to detect them programmatically.

In [ ]:
def troubleshoot():
    """Check for common ONNX setup issues."""
    issues = []

    # 1. Version compatibility
    onnx_ver = tuple(int(x) for x in onnx.__version__.split('.')[:2])
    ort_ver = tuple(int(x) for x in ort.__version__.split('.')[:2])

    if abs(onnx_ver[1] - ort_ver[1]) > 3:
        issues.append(
            f'ONNX ({onnx.__version__}) and ORT ({ort.__version__}) '
            f'version gap is large — consider aligning them')

    # 2. Protobuf version
    try:
        import google.protobuf
        pb_ver = google.protobuf.__version__
        pb_major = int(pb_ver.split('.')[0])
        if pb_major < 3:
            issues.append(f'protobuf {pb_ver} is too old (need >= 3.x)')
    except ImportError:
        issues.append('protobuf not found')

    # 3. Can build and run a model?
    try:
        X = make_tensor_value_info('X', TensorProto.FLOAT, [1])
        Y = make_tensor_value_info('Y', TensorProto.FLOAT, [1])
        g = make_graph([make_node('Relu', ['X'], ['Y'])], 'test', [X], [Y])
        m = make_model(g, opset_imports=[make_opsetid('', 17)])
        sess = ort.InferenceSession(
            m.SerializeToString(), providers=['CPUExecutionProvider'])
        r = sess.run(None, {'X': np.array([-1.0], dtype=np.float32)})[0]
        assert r[0] == 0.0
    except Exception as e:
        issues.append(f'Basic inference failed: {e}')

    # Report
    if issues:
        print('Issues found:')
        for i, issue in enumerate(issues, 1):
            print(f'  {i}. {issue}')
    else:
        print('No issues found — environment is healthy!')
    return len(issues) == 0

troubleshoot()

## Section 8: Challenge — Exportable Environment Report

In [ ]:
def generate_env_report():
    """Generate a complete environment report."""
    from datetime import datetime

    w = 60
    lines = []
    lines.append('=' * w)
    lines.append('ONNX ENVIRONMENT REPORT'.center(w))
    lines.append(f'Generated: {datetime.now().isoformat()}'.center(w))
    lines.append('=' * w)

    lines.append('\n[System]')
    lines.append(f'  OS:       {platform.system()} {platform.release()}')
    lines.append(f'  Arch:     {platform.machine()}')
    lines.append(f'  Python:   {sys.version.split()[0]}')
    lines.append(f'  NumPy:    {np.__version__}')

    lines.append('\n[ONNX]')
    lines.append(f'  onnx:            {onnx.__version__}')
    lines.append(f'  onnxruntime:     {ort.__version__}')
    lines.append(f'  Default opset:   {onnx_opset_version()}')
    lines.append(f'  IR version:      {onnx.IR_VERSION}')

    lines.append('\n[Execution Providers]')
    for ep in ort.get_available_providers():
        lines.append(f'  {ep}')

    lines.append('\n[Available Converters]')
    for name, mod in [('torch', 'torch'), ('tf2onnx', 'tf2onnx'),
                      ('skl2onnx', 'skl2onnx'), ('onnxmltools', 'onnxmltools')]:
        try:
            m = __import__(mod)
            lines.append(f'  {name:20s} v{m.__version__}')
        except ImportError:
            lines.append(f'  {name:20s} not installed')

    schemas = get_all_schemas_with_history()
    std_ops = len({s.name for s in schemas if s.domain == ''})
    lines.append(f'\n[Operators]')
    lines.append(f'  Standard (ai.onnx): {std_ops} unique operators')

    lines.append('\n[Verification]')
    try:
        X = make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
        Y = make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
        g = make_graph([make_node('Relu', ['X'], ['Y'])], 'v', [X], [Y])
        m = make_model(g, opset_imports=[make_opsetid('', 17)])
        check_model(m)
        s = ort.InferenceSession(
            m.SerializeToString(), providers=['CPUExecutionProvider'])
        s.run(None, {'X': np.ones((1, 4), dtype=np.float32)})
        lines.append('  Build → Validate → Run: PASS')
    except Exception as e:
        lines.append(f'  Build → Validate → Run: FAIL ({e})')

    lines.append('\n' + '=' * w)

    report = '\n'.join(lines)
    print(report)
    return report

report = generate_env_report()

---

## Summary

| Exercise | Topic | Key Takeaway |
|----------|-------|-------------|
| 1 | Version check | Always verify ONNX/ORT/protobuf versions |
| 2 | Verification script | Build → validate → run minimal model |
| 3 | Operator exploration | 170+ ops at current opset |
| 4 | EP capabilities | Test each execution provider |
| 5 | Full pipeline | Build → shape inference → save → load → run |
| 6 | Troubleshooting | Detect common setup issues |
| Challenge | Environment report | Exportable diagnostic summary |

**Next:** [ONNX Architecture and Internals →](../../03_ONNX_Architecture_and_Internals/)